In [1]:
jobs = {
    "J1": {
        "arrival": 0,
        "operations": [
            ("O11", "M1", 50),
            ("O12", "M2", 25)
        ]
    },
    "J2": {
        "arrival": 10,
        "operations": [
            ("O21", "M2", 30),
            ("O22", "M1", 35)
        ]
    },
    "J3": {
        "arrival": 20,
        "operations": [
            ("O31", "M1", 40),
            ("O32", "M2", 20)
        ]
    }
}


In [2]:
def create_fcfs_schedule(jobs):
    machine_available = {
        "M1": 0,
        "M2": 0
    }

    # Next operation index for each job
    next_operation = {
        job: 0 for job in jobs
    }

    # When the next operation of each job becomes ready
    operation_ready_time = {
        job: jobs[job]["arrival"] for job in jobs
    }

    schedule = []

    total_operations = sum(
        len(jobs[job]["operations"]) for job in jobs
    )

    current_time = 0

    while len(schedule) < total_operations:

        # Find operations that are ready
        ready_operations = []

        for job in jobs:
            index = next_operation[job]

            # Job still has operations remaining
            if index < len(jobs[job]["operations"]):

                operation, machine, processing_time = jobs[job]["operations"][index]

                # Operation is ready
                if operation_ready_time[job] <= current_time:
                    ready_operations.append({
                        "job": job,
                        "index": index,
                        "operation": operation,
                        "machine": machine,
                        "processing_time": processing_time,
                        "ready_time": operation_ready_time[job]
                    })

        # If nothing is ready, jump to the next operation arrival
        if not ready_operations:
            current_time = min(
                operation_ready_time[job]
                for job in jobs
                if next_operation[job] < len(jobs[job]["operations"])
                and operation_ready_time[job] > current_time
            )
            
            continue

        # FCFS:
        # Choose the operation that became ready first
        ready_operations.sort(key=lambda x: x["ready_time"])

        # Try to schedule the first FCFS operation
        selected = ready_operations[0]

        job = selected["job"]
        index = selected["index"]
        operation = selected["operation"]
        machine = selected["machine"]
        processing_time = selected["processing_time"]

        # The operation can only start when both
        # the operation and its machine are ready
        start_time = max(
            selected["ready_time"],
            machine_available[machine],
            current_time
        )

        finish_time = start_time + processing_time

        schedule.append({
            "job": job,
            "operation": operation,
            "machine": machine,
            "start": start_time,
            "finish": finish_time
        })

        # Update machine
        machine_available[machine] = finish_time

        # This operation is complete,
        # so the next operation of this job becomes ready
        next_operation[job] += 1
        operation_ready_time[job] = finish_time

        # Move time forward
        current_time = start_time

    return schedule

In [3]:
fcfs_schedule = create_fcfs_schedule(jobs)

for action in fcfs_schedule:
    print(action)

{'job': 'J1', 'operation': 'O11', 'machine': 'M1', 'start': 0, 'finish': 50}
{'job': 'J2', 'operation': 'O21', 'machine': 'M2', 'start': 10, 'finish': 40}
{'job': 'J3', 'operation': 'O31', 'machine': 'M1', 'start': 50, 'finish': 90}
{'job': 'J2', 'operation': 'O22', 'machine': 'M1', 'start': 90, 'finish': 125}
{'job': 'J1', 'operation': 'O12', 'machine': 'M2', 'start': 90, 'finish': 115}
{'job': 'J3', 'operation': 'O32', 'machine': 'M2', 'start': 115, 'finish': 135}


In [4]:
def spt_schedule(jobs):
    machine_available = {"M1": 0, "M2": 0}
    job_available = {job: data["arrival"] for job, data in jobs.items()}
    next_op = {job: 0 for job in jobs}

    schedule = []
    total_operations = sum(len(j["operations"]) for j in jobs.values())

    current_time = 0

    while len(schedule) < total_operations:

        candidates = []

        for job, data in jobs.items():
            i = next_op[job]

            if i >= len(data["operations"]):
                continue

            op, machine, processing_time = data["operations"][i]

            # Operation must be available NOW
            if (job_available[job] <= current_time and
                    machine_available[machine] <= current_time):

                candidates.append(
                    (processing_time, job, op, machine)
                )

        # If nothing is available, move time forward
        if not candidates:
            next_times = []

            for job, data in jobs.items():
                i = next_op[job]

                if i >= len(data["operations"]):
                    continue

                op, machine, processing_time = data["operations"][i]

                next_times.append(job_available[job])
                next_times.append(machine_available[machine])

            current_time = min(
                t for t in next_times if t > current_time
            )

            continue

        # SPT
        candidates.sort(key=lambda x: x[0])

        processing_time, job, op, machine = candidates[0]

        start = current_time
        end = start + processing_time

        schedule.append({
            "job": job,
            "operation": op,
            "machine": machine,
            "start": start,
            "end": end
        })

        machine_available[machine] = end
        job_available[job] = end
        next_op[job] += 1

        # Current time stays at the scheduling point
        # because another machine may still be available
        current_time = min(machine_available.values())

    return schedule


schedule = spt_schedule(jobs)

for action in schedule:
    print(action)

{'job': 'J1', 'operation': 'O11', 'machine': 'M1', 'start': 0, 'end': 50}
{'job': 'J2', 'operation': 'O21', 'machine': 'M2', 'start': 10, 'end': 40}
{'job': 'J1', 'operation': 'O12', 'machine': 'M2', 'start': 50, 'end': 75}
{'job': 'J2', 'operation': 'O22', 'machine': 'M1', 'start': 50, 'end': 85}
{'job': 'J3', 'operation': 'O31', 'machine': 'M1', 'start': 85, 'end': 125}
{'job': 'J3', 'operation': 'O32', 'machine': 'M2', 'start': 125, 'end': 145}
